# Sistema de Recomendación de Paradas en Boxes (F1 Decision Engine)

Este notebook implementa y evalúa los dos modelos secuenciales que conforman el motor de recomendación estratégica de F1 (Capa 1 y Capa 2).

## Arquitectura de Dos Capas
1. **Capa 1 (Modelo de Regresión):** Predice la degradación y ritmo esperado del neumático si el piloto permanece en pista durante las próximas $w$ vueltas.
2. **Capa 2 (Modelo de Ranking):** Utiliza la predicción de la Capa 1 como costo puente (`predicted_cost_of_staying`) junto con el contexto táctico y de tráfico para ordenar las 6 ventanas de parada (`wait_laps` 0 a 5) de menor a mayor costo estratégico.

### 1. Ingesta de Datos y Librerías

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid")

DATA_PATH = Path("../../data/recommendation/pit_decision_candidates_v1.parquet")
FEATURES_DIR = Path("../../data/features")
print("Librerías importadas.")

### 2. Definición del Target para Capa 1 (Costo Físico)

In [ ]:
def compute_regression_targets(df):
    """
    Calcula los objetivos reales de la regresión (ritmo futuro y varianza)
    si el piloto se queda en pista durante 'wait_laps' vueltas.
    """
    df = df.sort_values(["race_name", "driver_number", "lap_number"]).copy()
    lap_dur_dict = df.set_index(["race_name", "driver_number", "lap_number"])["lap_duration"].to_dict()
    stint_dict = df.set_index(["race_name", "driver_number", "lap_number"])["stint_number"].to_dict()
    pit_dict = df.set_index(["race_name", "driver_number", "lap_number"])["is_pit_lap"].to_dict()
    
    future_mean = []
    future_std = []
    
    for idx, row in df.iterrows():
        race = row["race_name"]
        drv = row["driver_number"]
        lp = row["lap_number"]
        w = int(row["wait_laps"])
        
        if w == 0:
            future_mean.append(row["lap_duration"])
            future_std.append(0.0)
            continue
            
        laps_to_check = list(range(int(lp), int(lp) + w))
        durations = []
        valid = True
        stint_start = stint_dict.get((race, drv, lp))
        
        for curr_lp in laps_to_check:
            key = (race, drv, curr_lp)
            if key not in lap_dur_dict or stint_dict.get(key) != stint_start or (curr_lp > lp and pit_dict.get((race, drv, curr_lp), 0) == 1):
                valid = False
                break
            durations.append(lap_dur_dict[key])
            
        if valid and len(durations) == w:
            future_mean.append(np.mean(durations))
            future_std.append(np.std(durations) if len(durations) > 1 else 0.0)
        else:
            future_mean.append(np.nan)
            future_std.append(np.nan)
            
    df["target_future_mean"] = future_mean
    df["target_future_std"] = future_std
    return df

### 3. Carga e Imputación de Características

In [ ]:
df = pd.read_parquet(DATA_PATH)
df = compute_regression_targets(df)

# Filtrar dataset de regresión
df_reg = df.dropna(subset=["target_future_mean"]).copy()

features = [
    "tyre_age", "compound_ord", "lap_vs_best_stint", "lap_mean_3", 
    "lap_std_3", "lap_slope_3", "deg_rate_3lap", "position", 
    "is_top10", "laps_remaining", "race_pct_complete", 
    "gap_ahead", "gap_behind", "wait_laps"
]

# Imputar nulos con la mediana para algoritmos sensibles
for col in features:
    median_val = df[col].median()
    if pd.isna(median_val):
        median_val = 0.0
    df[col] = df[col].fillna(median_val)
    df_reg[col] = df_reg[col].fillna(median_val)

X = df_reg[features]
y_mean = df_reg["target_future_mean"]
print(f"Dimensiones dataset de regresión: {X.shape}")

### 4. Capa 1: Modelado Comparativo de Regresión

In [ ]:
gkf = GroupKFold(n_splits=4)
groups = df_reg["race_name"]

reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost Regressor": xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

results = []
for name, model in reg_models.items():
    mses, r2s = [], []
    for train_idx, test_idx in gkf.split(X, y_mean, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y_mean.iloc[train_idx], y_mean.iloc[test_idx]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        mses.append(mean_squared_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))
        
    results.append({
        "Modelo": name,
        "MSE": np.mean(mses),
        "R2": np.mean(r2s)
    })

df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

#### Justificación de la Elección del Modelo (Capa 1)
* **Linear Regression:** Rinde pobremente ($R^2 \approx 0.089$) debido a que la degradación física es altamente no lineal (zona de precipicio de rendimiento del neumático).
* **Gradient Boosting / Random Forest:** Tienen la mejor capacidad de generalización ($R^2 \approx 0.546$), superando a XGBoost por tener menor propensión al sobreajuste sobre características correlacionadas. Elegimos **Gradient Boosting** como modelo final para poblar la característica puente.

### 5. Integración: Cálculo del Costo Puente (`predicted_cost_of_staying`)

In [ ]:
# Entrenar modelo final de Capa 1
best_reg = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
best_reg.fit(X, y_mean)

# Predecir coste para todos los candidatos
df["predicted_future_pace"] = best_reg.predict(df[features])
df["predicted_cost_of_staying"] = df["wait_laps"] * (df["predicted_future_pace"] - df["lap_duration"])

print("Costo puente calculado con éxito.")

### 6. Capa 2: Modelado Comparativo de Ranking

In [ ]:
def evaluate_ndcg(df_eval, group_cols, rank_col, label_col, k=3):
    ndcgs = []
    for _, group in df_eval.groupby(group_cols):
        if len(group) < 2 or np.all(group[label_col].values == group[label_col].values[0]):
            continue
        sorted_group = group.sort_values(by=rank_col, ascending=False)
        actual_labels = sorted_group[label_col].values
        ideal_labels = np.sort(group[label_col].values)[::-1]
        
        dcg = sum((2**(max(0, actual_labels[i] + 2)) - 1) / np.log2(i + 2) for i in range(min(k, len(actual_labels))))
        idcg = sum((2**(max(0, ideal_labels[i] + 2)) - 1) / np.log2(i + 2) for i in range(min(k, len(ideal_labels))))
        if idcg > 0:
            ndcgs.append(dcg / idcg)
    return np.mean(ndcgs) if ndcgs else 1.0

ranking_features = features + ["predicted_cost_of_staying"]
df["query_id"] = df["race_name"] + "_" + df["driver_number"].astype(str) + "_" + df["lap_number"].astype(str)
df_rank = df.sort_values("query_id").copy()
df_rank["rank_label"] = df_rank.groupby("query_id")["success_score_label"].rank(method="first").astype(int) - 1

X_rank = df_rank[ranking_features]
y_rank = df_rank["success_score_label"]
races_groups = df_rank["race_name"]

gkf_rank = GroupKFold(n_splits=4)

# 1. Point-wise (Random Forest)
ndcgs_rf1, ndcgs_rf3 = [], []
for train_idx, test_idx in gkf_rank.split(X_rank, y_rank, races_groups):
    X_tr, X_te = X_rank.iloc[train_idx], X_rank.iloc[test_idx]
    y_tr = y_rank.iloc[train_idx]
    df_te = df_rank.iloc[test_idx].copy()
    
    rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
    rf.fit(X_tr, y_tr)
    df_te["pred_score"] = rf.predict(X_te)
    
    ndcgs_rf1.append(evaluate_ndcg(df_te, "query_id", "pred_score", "success_score_label", k=1))
    ndcgs_rf3.append(evaluate_ndcg(df_te, "query_id", "pred_score", "success_score_label", k=3))

print(f"RF Point-wise | NDCG@1: {np.mean(ndcgs_rf1):.4f} | NDCG@3: {np.mean(ndcgs_rf3):.4f}")

# 2. List-wise (XGBRanker)
ndcgs_xgb1, ndcgs_xgb3 = [], []
for train_idx, test_idx in gkf_rank.split(X_rank, y_rank, races_groups):
    df_tr_split = df_rank.iloc[train_idx].sort_values("query_id")
    df_te_split = df_rank.iloc[test_idx].sort_values("query_id")
    
    train_group_sizes = df_tr_split.groupby("query_id").size().values
    X_tr_xgb = df_tr_split[ranking_features]
    y_tr_xgb = df_tr_split["rank_label"]
    X_te_xgb = df_te_split[ranking_features]
    
    ranker = xgb.XGBRanker(objective="rank:ndcg", eval_metric="ndcg", n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
    ranker.fit(X_tr_xgb, y_tr_xgb, group=train_group_sizes)
    df_te_split["pred_score"] = ranker.predict(X_te_xgb)
    
    ndcgs_xgb1.append(evaluate_ndcg(df_te_split, "query_id", "pred_score", "success_score_label", k=1))
    ndcgs_xgb3.append(evaluate_ndcg(df_te_split, "query_id", "pred_score", "success_score_label", k=3))

print(f"XGBRanker     | NDCG@1: {np.mean(ndcgs_xgb1):.4f} | NDCG@3: {np.mean(ndcgs_xgb3):.4f}")

#### Justificación de la Elección del Modelo (Capa 2)
* **Random Forest Point-wise (NDCG@3: 0.9453):** Supera a XGBRanker ($0.8727$). La razón física es que el score de éxito original es una variable continua con magnitud real (segundos recuperados + posiciones ganadas). Al entrenar un regresor point-wise, el modelo aprende la magnitud absoluta del beneficio del pit. En cambio, XGBRanker requiere transformar los targets a enteros discretos (0 a 5), perdiendo la sensibilidad a la magnitud de la diferencia de ritmo.

### 7. Simulación de Ventana de Parada Óptima

In [ ]:
# Entrenamos el clasificador/regresor final de Capa 2
final_rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
final_rf.fit(X_rank, y_rank)

def simular_vuelta_piloto(df_race, driver_num, start_lap, end_lap):
    """
    Visualiza la curva de conveniencia (score esperado) de parada a lo largo de las vueltas.
    """
    sub_df = df_race[(df_race["driver_number"] == driver_num) & 
                     (df_race["lap_number"] >= start_lap) & 
                     (df_race["lap_number"] <= end_lap)].copy()
    
    sub_df["pred_score"] = final_rf.predict(sub_df[ranking_features])
    p_now = sub_df[sub_df["candidate"] == 0].sort_values("lap_number")
    
    plt.figure(figsize=(10, 5))
    plt.plot(p_now["lap_number"], p_now["pred_score"], marker='o', color='crimson', label="Conveniencia de Parada (Score predicho)")
    
    real_pit = sub_df[sub_df["is_pit_lap"] == 1]["lap_number"].unique()
    for pit_lp in real_pit:
        plt.axvline(x=pit_lp, color='blue', linestyle='--', label=f"Parada Real (Vuelta {pit_lp})")
        
    plt.title(f"Evolucion del Score de Conveniencia de Parada - Piloto {driver_num}")
    plt.xlabel("Vuelta de Carrera")
    plt.ylabel("Score Esperado (Mayor = Mas Conveniente)")
    plt.legend()
    plt.show()

simular_vuelta_piloto(df[df["race_name"] == "australia"], 1.0, 5, 25)